# 10. Create the new frozen final-evaluation split

This notebook creates **one new shared split** for every final model rerun.

Default proportions are kept at **60% model training / 20% threshold validation / 20% final test**. The split uses a new seed rather than the earlier `random_state=42`, so the final-test set is not the same old 20% set.

After this notebook is run, do not regenerate the split unless you intentionally restart the whole final-evaluation workflow.

In [5]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FINAL_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
FINAL_SPLIT_SEED = 20260823

assert 0 < FINAL_TEST_SIZE < 1
assert 0 < VALIDATION_SIZE < 1
assert FINAL_TEST_SIZE + VALIDATION_SIZE < 1

PROJECT_ROOT = Path.cwd()
for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Split output:", SPLIT_DIR)


Project root: /Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection
Dataset: /Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv
Split output: /Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Data_Splits


## Load the cleaned data

Only row positions and the binary target are needed to create the split. All model notebooks later reload the same source dataset and use these saved row positions.

In [6]:

df = pd.read_csv(DATA_PATH)

target_col = "readmitted_30"
if target_col not in df.columns:
    raise ValueError(f"Missing target column: {target_col}")

y = df[target_col].astype(int)

if not set(y.unique()).issubset({0, 1}):
    raise ValueError("Target must contain only 0/1.")

row_positions = np.arange(len(df), dtype=int)

print("Rows:", len(df))
print("Positive count:", int(y.sum()))
print("Positive rate:", float(y.mean()))


Rows: 69987
Positive count: 6285
Positive rate: 0.08980239187277637


## Create the frozen split

The final test set is selected first. A separate validation set is then selected from the remaining development data so that its **overall** size is 20%.

With the default settings this produces exactly 60/20/20.

In [7]:

development_idx, final_test_idx = train_test_split(
    row_positions,
    test_size=FINAL_TEST_SIZE,
    stratify=y,
    random_state=FINAL_SPLIT_SEED,
)

validation_fraction_within_development = (
    VALIDATION_SIZE / (1.0 - FINAL_TEST_SIZE)
)

model_train_idx, validation_idx = train_test_split(
    development_idx,
    test_size=validation_fraction_within_development,
    stratify=y.iloc[development_idx],
    random_state=FINAL_SPLIT_SEED,
)

# Safety checks
all_sets = [
    set(model_train_idx),
    set(validation_idx),
    set(final_test_idx),
]

assert all_sets[0].isdisjoint(all_sets[1])
assert all_sets[0].isdisjoint(all_sets[2])
assert all_sets[1].isdisjoint(all_sets[2])
assert len(model_train_idx) + len(validation_idx) + len(final_test_idx) == len(df)

split_summary = pd.DataFrame({
    "split": [
        "model_training",
        "threshold_validation",
        "final_test_LOCKED",
    ],
    "rows": [
        len(model_train_idx),
        len(validation_idx),
        len(final_test_idx),
    ],
    "fraction": [
        len(model_train_idx) / len(df),
        len(validation_idx) / len(df),
        len(final_test_idx) / len(df),
    ],
    "positive_count": [
        int(y.iloc[model_train_idx].sum()),
        int(y.iloc[validation_idx].sum()),
        int(y.iloc[final_test_idx].sum()),
    ],
    "positive_rate": [
        float(y.iloc[model_train_idx].mean()),
        float(y.iloc[validation_idx].mean()),
        float(y.iloc[final_test_idx].mean()),
    ],
})

split_summary


,split,rows,fraction,positive_count,positive_rate
0,model_training,41991,0.599983,3771,0.089805
1,threshold_validation,13998,0.200009,1257,0.089799
2,final_test_LOCKED,13998,0.200009,1257,0.089799


## Save the split

The final-test file is saved, but notebooks 11 to 16 deliberately never read it.

In [8]:

def save_rows(path, indices):
    pd.DataFrame({
        "row_position": np.asarray(indices, dtype=int)
    }).to_csv(path, index=False)

save_rows(SPLIT_DIR / "model_train_rows.csv", model_train_idx)
save_rows(SPLIT_DIR / "threshold_validation_rows.csv", validation_idx)
save_rows(SPLIT_DIR / "final_test_rows_LOCKED.csv", final_test_idx)

split_summary.to_csv(
    SPLIT_DIR / "split_summary.csv",
    index=False
)

dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

manifest = {
    "dataset_relative_path": "Processed_Dataset/diabetic_data_cleaned_stage1.csv",
    "dataset_sha256": dataset_sha256,
    "n_rows": int(len(df)),
    "target": target_col,
    "final_test_size": float(FINAL_TEST_SIZE),
    "validation_size": float(VALIDATION_SIZE),
    "model_training_size": float(1.0 - FINAL_TEST_SIZE - VALIDATION_SIZE),
    "final_split_seed": int(FINAL_SPLIT_SEED),
}

with open(SPLIT_DIR / "split_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved frozen split files.")
print("DO NOT regenerate them after starting final model tuning.")


Saved frozen split files.
DO NOT regenerate them after starting final model tuning.
